In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

# 1. MONTAR GOOGLE DRIVE
drive.mount('/content/drive')

# Rutas de entrada y salida
ruta_csv = '/content/drive/My Drive/Paderborn_QML/paderborn_features_final.csv'
ruta_salida = '/content/drive/My Drive/Paderborn_QML/paderborn_features_tratadas.csv'

# 2. CARGA DE DATOS
df = pd.read_csv(ruta_csv)
print(f"✅ Dataset cargado. Dimensiones iniciales: {df.shape}")

# 3. COMPROBACIÓN Y TRATAMIENTO DE NULOS E INFINITOS
# Reemplazar infinitos por NaN para evaluar todo junto
df = df.replace([np.inf, -np.inf], np.nan)

n_nulos = df.isnull().sum().sum()
print(f"\n🔍 Verificación de calidad:")
print(f"   • Valores nulos/infinitos detectados: {n_nulos}")

if n_nulos > 0:
    df = df.dropna().reset_index(drop=True)
    print(f"   ✅ Se eliminaron registros corruptos. Nuevas dimensiones: {df.shape}")
else:
    print("   ✅ El dataset está 100% libre de nulos e infinitos.")

# 4. IDENTIFICACIÓN DE COLUMNAS
op_cols = ['RPM', 'Torque_Nm', 'Fuerza_N']  # Variables de condición operativa (NO recortar)
stat_signal_cols = ['RMS', 'Kurtosis', 'Skewness', 'Crest_Factor']
fft_cols = [col for col in df.columns if col.startswith('fft_')]
target_col = [col for col in df.columns if col.lower() in ['label', 'target', 'state', 'fault', 'clase', 'condition']]

col_etiqueta = target_col[0] if target_col else 'label'

# Columnas de señal de vibración sobre las que SÍ aplica la limpieza de ruido/atípicos
signal_cols = stat_signal_cols + fft_cols

print(f"\n📊 Estructura identificada:")
print(f"   • Espectro FFT: {len(fft_cols)} columnas")
print(f"   • Columna objetivo: '{col_etiqueta}'")

# 5. TRANSFORMACIÓN LOGARÍTMICA EN FFT
# log1p -> log(1 + x) previene log(0) de forma matemáticamente segura
df[fft_cols] = np.log1p(np.abs(df[fft_cols]))
print(f"\n✅ Transformación logarítmica log(1+x) aplicada a las {len(fft_cols)} columnas FFT.")

# 6. SEPARACIÓN DE CLASES Y TRATAMIENTO DE ATÍPICOS (CLIPPING)
# Máscara para detectar la condición de motor sano
mask_sano = df[col_etiqueta].astype(str).str.contains('Normal|sano|healthy|0', case=False, regex=True)

df_sano = df[mask_sano].copy()
df_falla = df[~mask_sano].copy()

# Calcular percentiles 1% y 99% ÚNICAMENTE en las variables de señal de la clase SANA
p_lower = np.percentile(df_sano[signal_cols], 1, axis=0)
p_upper = np.percentile(df_sano[signal_cols], 99, axis=0)

# Acotar los extremos SOLAMENTE en las señales de vibración (preserva las condiciones de operación intactas)
df_sano[signal_cols] = np.clip(df_sano[signal_cols].values, p_lower, p_upper)
print("✅ Clipping (1%-99%) aplicado únicamente a las variables de señal (RMS, estadísticas y FFT).")

# 7. CONSOLIDACIÓN Y EXPORTACIÓN A DRIVE
df_tratado = pd.concat([df_sano, df_falla], axis=0).reset_index(drop=True)
df_tratado.to_csv(ruta_salida, index=False)

print("\n" + "="*70)
print("PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*70)
print(f"• Dimensiones finales: {df_tratado.shape}")
print(f"💾 Archivo guardado en Google Drive:")
print(f"   {ruta_salida}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset cargado. Dimensiones iniciales: (59867, 72)

🔍 Verificación de calidad:
   • Valores nulos/infinitos detectados: 0
   ✅ El dataset está 100% libre de nulos e infinitos.

📊 Estructura identificada:
   • Espectro FFT: 64 columnas
   • Columna objetivo: 'Label'

✅ Transformación logarítmica log(1+x) aplicada a las 64 columnas FFT.
✅ Clipping (1%-99%) aplicado únicamente a las variables de señal (RMS, estadísticas y FFT).

PROCESAMIENTO COMPLETADO EXITOSAMENTE
• Dimensiones finales: (59867, 72)
💾 Archivo guardado en Google Drive:
   /content/drive/My Drive/Paderborn_QML/paderborn_features_tratadas.csv
